# Data Cleaning


In [17]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
# Load the data
df = pd.read_csv('../data/hmeq.csv')

# Create a copy for cleaning 
df_clean = df.copy()

In [19]:
# Understand the missing pattern
missing_summary = pd.DataFrame({
    "Missing_count" : df.isnull().sum(),
    "Missing_percentage" : (df.isnull().sum() / len(df) * 100).round(2),
    "Data_type" : df.dtypes,
})
missing_summary = missing_summary.sort_values("Missing_percentage", ascending=False)
missing_summary


,Missing_count,Missing_percentage,Data_type
DEBTINC,1267,21.26,float64
DEROG,708,11.88,float64
DELINQ,580,9.73,float64
MORTDUE,518,8.69,float64
YOJ,515,8.64,float64
NINQ,510,8.56,float64
CLAGE,308,5.17,float64
JOB,279,4.68,str
REASON,252,4.23,str
CLNO,222,3.72,float64


In [20]:
# Create flags for missing values 
df_clean['DEBTINC_missing'] = df['DEBTINC'].isnull().astype(int)
df_clean['MORTDUE_missing'] = df['MORTDUE'].isnull().astype(int)
df_clean['VALUE_missing'] = df['VALUE'].isnull().astype(int)
df_clean.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,NaN,1,0,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,NaN,1,0,0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,NaN,1,0,0
3,1,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,NaN,1,0,0


In [21]:
# strategy for DEBTINC median imputation by JOB category
# first clean JOB
df_clean['JOB'] = df_clean['JOB'].fillna('Unknown')

# Impute DEBTINC by job cat
debtinc_by_job = df_clean.groupby("JOB")["DEBTINC"].median().round(2)
debtinc_by_job

JOB
Mgr        35.66
Office     36.16
Other      35.25
ProfExe    33.38
Sales      35.76
Self       34.83
Unknown    30.31
Name: DEBTINC, dtype: float64

In [22]:
df_clean['DEBTINC'] = df_clean.groupby('JOB')['DEBTINC'].transform(
    lambda x: x.fillna(x.median())
)
df_clean.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,35.247328,1,0,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,35.247328,1,0,0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,35.247328,1,0,0
3,1,1500,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,30.311902,1,1,1
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,36.158718,1,0,0


In [23]:
# MORTDUE and VALUE often missing together 
# Strategy median imputation 

print(f"   MORTDUE median: {df_clean['MORTDUE'].median()}")
print(f"   VALUE median: {df_clean['VALUE'].median()}")

df_clean['MORTDUE'] = df_clean['MORTDUE'].fillna(df_clean['MORTDUE'].median())
df_clean['VALUE'] = df_clean['VALUE'].fillna(df_clean['VALUE'].median())



   MORTDUE median: 65019.0
   VALUE median: 89235.5


In [24]:
# YOJ strategy median
df_clean['YOJ'] = df_clean['YOJ'].fillna(df_clean['YOJ'].median())

In [25]:
# REASONS categorical 
df_clean['REASON'] = df_clean['REASON'].fillna(df_clean['REASON'].mode()[0])


In [26]:
# Credit Behavior Variables (DEROG, DELINQ, CLAGE, NINQ, CLNO) median
df_clean['DEROG'] = df_clean['DEROG'].fillna(df_clean['DEROG'].median())
df_clean['DELINQ'] = df_clean['DELINQ'].fillna(df_clean['DELINQ'].median())
df_clean['CLAGE'] = df_clean['CLAGE'].fillna(df_clean['CLAGE'].median())
df_clean['NINQ'] = df_clean['NINQ'].fillna(df_clean['NINQ'].median())
df_clean['CLNO'] = df_clean['CLNO'].fillna(df_clean['CLNO'].median())

In [27]:
# verify missing data remaining 
remaining_missing = df_clean.isnull().sum().sum()
if remaining_missing == 0:
    print("All missing values successfully handled")
else:
    print(f"Remaining data to clea: {remaining_missing}")

All missing values successfully handled


In [28]:
# Save cleaned data
df_clean.to_csv('../data/hmeq_cleaned.csv', index=False)

# Feature Engineering

In [29]:
# Loan to value ratio (LTV)
# Critical for LGD calculations - measures collateral coverage

df_clean["LTV"] = df_clean["LOAN"] / df_clean["VALUE"]

# cap at 200% to handle outliers (some people borrow more than property worth)
df_clean['LTV'] = df_clean['LTV'].clip(upper=2.0)

print(f"   Mean LTV: {df_clean['LTV'].mean():.2%}")
print(f"   Median LTV: {df_clean['LTV'].median():.2%}")
print(f"   Max LTV: {df_clean['LTV'].max():.2%}")

   Mean LTV: 21.65%
   Median LTV: 17.05%
   Max LTV: 200.00%


In [30]:
# DEBT to Value ratio - measures total debt burden relative to property value
df_clean['DEBT_TO_VALUE'] = (df_clean['LOAN'] + df_clean['MORTDUE']) / df_clean['VALUE']
df_clean['DEBT_TO_VALUE'] = df_clean['DEBT_TO_VALUE'].clip(upper=3.0)

print(f"   Mean Debt-to-Value: {df_clean['DEBT_TO_VALUE'].mean():.2%}")

   Mean Debt-to-Value: 95.82%


In [31]:
# Credit utilisation indicator - combined deliquency and derogatory marks
df_clean['CREDIT_ISSUES'] = df_clean['DEROG'] + df_clean['DELINQ']

print(f"% of borrowers with no credit issues: {(df_clean['CREDIT_ISSUES'] == 0).mean():.1%}")
print(f"% with 1+ credit issues: {(df_clean['CREDIT_ISSUES'] > 0).mean():.1%}")
print(f"% with 3+ credit issues: {(df_clean['CREDIT_ISSUES'] >= 3).mean():.1%}")


% of borrowers with no credit issues: 72.6%
% with 1+ credit issues: 27.4%
% with 3+ credit issues: 8.4%


In [32]:
# high enquiry activity - recent enquiry can signal credit stress
df_clean['HIGH_ENQUIRIES'] = (df_clean['NINQ'] >= 2).astype(int)

print(f"% with high inquiry activity: {df_clean['HIGH_ENQUIRIES'].mean():.1%}")

% with high inquiry activity: 26.5%


In [33]:
# Employment Stability 
df_clean['JOB_STABILITY'] = pd.cut(
    df_clean['YOJ'], 
    bins=[0, 2, 10, 100],
    labels=['New', 'Stable', 'Established'],
    include_lowest=True
)

print(df_clean['JOB_STABILITY'].value_counts())

JOB_STABILITY
Stable         2955
Established    1782
New            1223
Name: count, dtype: int64


In [34]:
# Combine multiple risk factors - LTV > 80% or DEBTINC > 40% or CREDIT_ISSUES > 0
df_clean['HIGH_RISK_FLAG'] = (
    (df_clean['LTV'] > 0.80) | 
    (df_clean['DEBTINC'] > 40) | 
    (df_clean['CREDIT_ISSUES'] > 0)
).astype(int)

print(f"   % flagged as high-risk: {df_clean['HIGH_RISK_FLAG'].mean():.1%}")

   % flagged as high-risk: 39.4%


In [35]:
# Create default rate at high vs low risk segments
risk_segment_default = df_clean.groupby('HIGH_RISK_FLAG')['BAD'].agg(['mean', 'count'])
risk_segment_default.columns = ['Default_Rate', 'Count']
risk_segment_default['Default_Rate'] = risk_segment_default['Default_Rate'] * 100
risk_segment_default

,Default_Rate,Count
HIGH_RISK_FLAG,,
0,11.455451,3614
1,33.034953,2346


In [36]:
df_clean

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,...,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing,LTV,DEBT_TO_VALUE,CREDIT_ISSUES,HIGH_ENQUIRIES,JOB_STABILITY,HIGH_RISK_FLAG
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,...,35.247328,1,0,0,0.028187,0.690839,0.0,0,Established,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,...,35.247328,1,0,0,0.019006,1.043173,2.0,0,Stable,1
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,...,35.247328,1,0,0,0.089820,0.898204,0.0,0,Stable,0
3,1,1500,65019.0,89235.5,DebtCon,Unknown,7.0,0.0,0.0,173.466667,...,30.311902,1,1,1,0.016809,0.745432,0.0,0,Stable,0
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,...,36.158718,1,0,0,0.015179,0.888393,0.0,0,Stable,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5955,0,88900,57264.0,90185.0,DebtCon,Other,16.0,0.0,0.0,221.808718,...,36.112347,0,0,0,0.985752,1.620713,0.0,0,Established,1
5956,0,89000,54576.0,92937.0,DebtCon,Other,16.0,0.0,0.0,208.692070,...,35.859971,0,0,0,0.957638,1.544874,0.0,0,Established,1
5957,0,89200,54045.0,92924.0,DebtCon,Other,15.0,0.0,0.0,212.279697,...,35.556590,0,0,0,0.959924,1.541529,0.0,0,Established,1
5958,0,89800,50370.0,91861.0,DebtCon,Other,14.0,0.0,0.0,213.892709,...,34.340882,0,0,0,0.977564,1.525892,0.0,0,Established,1


In [38]:
# save to new csv
df_clean.to_csv('../data/hmeq_features.csv', index=False)